# Credit Card Fraud Detection — EDA & Model Evaluation

**Project:** FraudGuard Anomaly Detection API  
**Model:** Isolation Forest (unsupervised)  
**Dataset:** Kaggle Credit Card Fraud Detection

This notebook covers:
1. Dataset overview & class imbalance
2. Feature distributions (normal vs fraud)
3. Correlation analysis
4. Model training & threshold tuning
5. ROC curve & AUC
6. Precision-Recall curve
7. Confusion matrix
8. Anomaly score distributions
9. Top discriminative features

## 0. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333',
    'axes.labelcolor':  '#ccc',
    'text.color':       '#ccc',
    'xtick.color':      '#999',
    'ytick.color':      '#999',
    'grid.color':       '#2a2a2a',
    'grid.linestyle':   '--',
    'figure.dpi':       120,
})

CYAN   = '#00d4ff'
PURPLE = '#b44fff'
RED    = '#ff4f4f'
GREEN  = '#4fff91'
ORANGE = '#ff9f4f'

FEATURE_COLS = [
    'Time', 'V1','V2','V3','V4','V5','V6','V7','V8','V9','V10',
    'V11','V12','V13','V14','V15','V16','V17','V18','V19','V20',
    'V21','V22','V23','V24','V25','V26','V27','V28','Amount'
]
print('Setup complete.')

## 1. Load Dataset

In [ ]:
DATA_PATH = Path('../data/creditcard.csv')
df = pd.read_csv(DATA_PATH)

print(f'Shape         : {df.shape}')
print(f'Total txns    : {len(df):,}')
print(f'Fraud txns    : {df["Class"].sum():,}  ({df["Class"].mean()*100:.3f}%)')
print(f'Normal txns   : {(df["Class"]==0).sum():,}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Time range    : {df["Time"].min():.0f}s - {df["Time"].max():.0f}s ({df["Time"].max()/3600:.1f} hrs)')
print(f'Amount range  : ${df["Amount"].min():.2f} - ${df["Amount"].max():,.2f}')
df.describe().T

## 2. Class Imbalance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['Class'].value_counts()
axes[0].bar(['Normal', 'Fraud'], counts.values, color=[GREEN, RED], width=0.5)
axes[0].set_yscale('log')
axes[0].set_title('Transaction Class Distribution (log scale)', color='white')
axes[0].set_ylabel('Count')
for i, (label, count) in enumerate(zip(['Normal', 'Fraud'], counts.values)):
    axes[0].text(i, count * 1.3, f'{count:,}', ha='center', color='white', fontsize=10)

axes[1].pie(
    counts.values,
    labels=[f'Normal\n{counts.values[0]/len(df)*100:.2f}%',
            f'Fraud\n{counts.values[1]/len(df)*100:.3f}%'],
    colors=[GREEN, RED],
    startangle=90,
    wedgeprops=dict(edgecolor='#0f0f0f', linewidth=2),
    textprops={'color': 'white'}
)
axes[1].set_title('Class Proportion', color='white')

plt.suptitle('Severe Class Imbalance — Why Unsupervised Learning Makes Sense',
             color=CYAN, fontsize=12)
plt.tight_layout()
plt.savefig('class_imbalance.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Transaction Amount & Time: Normal vs Fraud

In [ ]:
normal = df[df['Class'] == 0]
fraud  = df[df['Class'] == 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(normal['Amount'], bins=100, color=GREEN, alpha=0.5, label='Normal', density=True)
axes[0].hist(fraud['Amount'],  bins=50,  color=RED,   alpha=0.8, label='Fraud',  density=True)
axes[0].set_xlim(0, 1000)
axes[0].axvline(fraud['Amount'].median(),  color=RED,   linestyle='--', label=f'Fraud median:  \${fraud["Amount"].median():.2f}')
axes[0].axvline(normal['Amount'].median(), color=GREEN, linestyle='--', label=f'Normal median: \${normal["Amount"].median():.2f}')
axes[0].set_title('Transaction Amount Distribution', color='white')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=8)

axes[1].hist(normal['Time']/3600, bins=100, color=GREEN, alpha=0.5, label='Normal', density=True)
axes[1].hist(fraud['Time']/3600,  bins=50,  color=RED,   alpha=0.8, label='Fraud',  density=True)
axes[1].set_title('Transaction Time Distribution', color='white')
axes[1].set_xlabel('Hours from first transaction')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=8)

plt.suptitle('Amount & Time: Normal vs Fraud', color=CYAN, fontsize=12)
plt.tight_layout()
plt.savefig('amount_time_dist.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'Normal - Amount mean: \${normal["Amount"].mean():.2f}  median: \${normal["Amount"].median():.2f}')
print(f'Fraud  - Amount mean: \${fraud["Amount"].mean():.2f}  median: \${fraud["Amount"].median():.2f}')

## 4. V-Feature Distributions: Normal vs Fraud

V1–V28 are PCA-transformed features. The ones with the most separation between classes are most useful to the model.

In [ ]:
v_features = [f'V{i}' for i in range(1, 29)]

separation = {}
for col in v_features:
    diff = abs(normal[col].mean() - fraud[col].mean())
    pooled_std = np.sqrt((normal[col].std()**2 + fraud[col].std()**2) / 2)
    separation[col] = diff / pooled_std

top8 = sorted(separation, key=separation.get, reverse=True)[:8]
print('Top 8 most discriminative features:', top8)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(top8):
    axes[i].hist(normal[col], bins=80, color=GREEN, alpha=0.5, label='Normal', density=True)
    axes[i].hist(fraud[col],  bins=50, color=RED,   alpha=0.7, label='Fraud',  density=True)
    axes[i].set_title(f'{col}  (sep={separation[col]:.2f})', color='white', fontsize=9)
    axes[i].set_xlabel('Value', fontsize=8)
    axes[i].tick_params(labelsize=7)
    if i == 0:
        axes[i].legend(fontsize=8)

plt.suptitle('Top 8 Most Discriminative V-Features', color=CYAN, fontsize=13)
plt.tight_layout()
plt.savefig('feature_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Feature Correlation with Fraud Label

In [ ]:
corr_with_class = df[FEATURE_COLS + ['Class']].corr()['Class'].drop('Class').sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = [RED if v < 0 else GREEN for v in corr_with_class.values]
axes[0].barh(corr_with_class.index, corr_with_class.values, color=colors, edgecolor='none')
axes[0].axvline(0, color='white', linewidth=0.5)
axes[0].set_title('Feature Correlation with Fraud Label', color='white')
axes[0].set_xlabel('Pearson Correlation')
axes[0].tick_params(labelsize=8)

top_corr_features = corr_with_class.abs().nlargest(12).index.tolist()
corr_matrix = df[top_corr_features + ['Class']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, ax=axes[1], mask=mask,
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    linewidths=0.3, linecolor='#0f0f0f',
    cbar_kws={'shrink': 0.8}
)
axes[1].set_title('Correlation Matrix — Top 12 Features', color='white')
axes[1].tick_params(labelsize=8)

plt.suptitle('Feature Correlations', color=CYAN, fontsize=13)
plt.tight_layout()
plt.savefig('correlations.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Train Isolation Forest

Trained **only on normal transactions** — the model learns what legitimate behaviour looks like, then flags deviations.

In [ ]:
import time

X_normal = normal[FEATURE_COLS].values
X_all    = df[FEATURE_COLS].values
y_true   = df['Class'].values

scaler = StandardScaler()
X_normal_scaled = scaler.fit_transform(X_normal)
X_all_scaled    = scaler.transform(X_all)

print('Training Isolation Forest...')
t0 = time.time()
iso = IsolationForest(n_estimators=200, contamination=0.002, random_state=42, n_jobs=-1)
iso.fit(X_normal_scaled)
print(f'Done in {time.time()-t0:.1f}s')

scores = iso.decision_function(X_all_scaled)
normal_scores = scores[y_true == 0]
fraud_scores  = scores[y_true == 1]

print(f'\nScore range   : {scores.min():.4f} -> {scores.max():.4f}')
print(f'Normal mean   : {normal_scores.mean():.4f}')
print(f'Fraud  mean   : {fraud_scores.mean():.4f}')

## 7. Anomaly Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

threshold = np.percentile(normal_scores, 1)

axes[0].hist(normal_scores, bins=120, color=GREEN, alpha=0.5, density=True, label=f'Normal (n={len(normal_scores):,})')
axes[0].hist(fraud_scores,  bins=60,  color=RED,   alpha=0.8, density=True, label=f'Fraud  (n={len(fraud_scores):,})')
axes[0].axvline(threshold, color=ORANGE, linestyle='--', linewidth=1.5, label=f'Threshold: {threshold:.4f}')
axes[0].set_title('Anomaly Score Distribution', color='white')
axes[0].set_xlabel('Score (lower = more anomalous)')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9)

bp = axes[1].boxplot(
    [normal_scores, fraud_scores],
    labels=['Normal', 'Fraud'],
    patch_artist=True,
    medianprops=dict(color='white', linewidth=2),
    whiskerprops=dict(color='#666'),
    capprops=dict(color='#666'),
    flierprops=dict(marker='.', markersize=2, alpha=0.3)
)
bp['boxes'][0].set_facecolor(GREEN); bp['boxes'][0].set_alpha(0.6)
bp['boxes'][1].set_facecolor(RED);   bp['boxes'][1].set_alpha(0.7)
axes[1].set_title('Score by Class (Box Plot)', color='white')
axes[1].set_ylabel('Isolation Forest Score')

plt.suptitle('Anomaly Scores: Model Separates Fraud from Normal', color=CYAN, fontsize=12)
plt.tight_layout()
plt.savefig('score_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

## 8. ROC Curve & AUC

In [ ]:
scores_for_roc = -scores  # negate: higher = more fraudulent

fpr, tpr, roc_thresholds = roc_curve(y_true, scores_for_roc)
auc = roc_auc_score(y_true, scores_for_roc)

optimal_idx = np.argmax(tpr - fpr)
opt_fpr, opt_tpr = fpr[optimal_idx], tpr[optimal_idx]

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr, tpr, color=CYAN, linewidth=2.5, label=f'Isolation Forest (AUC = {auc:.4f})')
ax.plot([0,1],[0,1], color='#555', linestyle='--', linewidth=1, label='Random classifier')
ax.scatter(opt_fpr, opt_tpr, color=ORANGE, s=120, zorder=5,
           label=f'Optimal point\n(TPR={opt_tpr:.3f}, FPR={opt_fpr:.4f})')
ax.fill_between(fpr, tpr, alpha=0.08, color=CYAN)
ax.set_xlim([-0.01, 1.01]); ax.set_ylim([-0.01, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curve — Isolation Forest\nAUC = {auc:.4f}', color='white', pad=12)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)

axins = ax.inset_axes([0.08, 0.45, 0.42, 0.42])
axins.plot(fpr, tpr, color=CYAN, linewidth=2)
axins.scatter(opt_fpr, opt_tpr, color=ORANGE, s=60, zorder=5)
axins.set_xlim(0, 0.1); axins.set_ylim(0.5, 1.0)
axins.set_title('Zoomed (FPR 0-0.1)', fontsize=7, color='white')
axins.tick_params(labelsize=7)
ax.indicate_inset_zoom(axins, edgecolor=ORANGE)

plt.tight_layout()
plt.savefig('roc_curve.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'AUC-ROC: {auc:.4f}')
print(f'At optimal threshold -> TPR: {opt_tpr:.4f}  FPR: {opt_fpr:.4f}')

## 9. Precision-Recall Curve

With severe class imbalance, **Precision-Recall** is more informative than ROC alone.

In [ ]:
precision, recall, pr_thresholds = precision_recall_curve(y_true, scores_for_roc)
ap = average_precision_score(y_true, scores_for_roc)
baseline = y_true.mean()

f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-8)
best_f1_idx = np.argmax(f1_scores)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(recall, precision, color=PURPLE, linewidth=2.5, label=f'Isolation Forest (AP={ap:.4f})')
axes[0].axhline(baseline, color='#555', linestyle='--', label=f'Random baseline = {baseline:.4f}')
axes[0].scatter(recall[best_f1_idx], precision[best_f1_idx], color=ORANGE, s=120, zorder=5,
                label=f'Best F1={f1_scores[best_f1_idx]:.3f}')
axes[0].fill_between(recall, precision, alpha=0.08, color=PURPLE)
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title(f'Precision-Recall Curve\nAP = {ap:.4f}', color='white')
axes[0].legend(fontsize=9)

axes[1].plot(pr_thresholds, f1_scores, color=CYAN, linewidth=2)
axes[1].axvline(pr_thresholds[best_f1_idx], color=ORANGE, linestyle='--',
                label=f'Best threshold: {pr_thresholds[best_f1_idx]:.4f}')
axes[1].set_xlabel('Decision Threshold'); axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 Score vs Threshold', color='white')
axes[1].legend(fontsize=9)

plt.suptitle('Precision-Recall Analysis', color=CYAN, fontsize=13)
plt.tight_layout()
plt.savefig('precision_recall.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'Average Precision : {ap:.4f}')
print(f'Best F1 Score     : {f1_scores[best_f1_idx]:.4f}')

## 10. Confusion Matrix

In [ ]:
api_threshold = np.percentile(normal_scores, 1)
y_pred = (scores < api_threshold).astype(int)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay(cm, display_labels=['Normal','Fraud']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (counts)', color='white')

cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
ConfusionMatrixDisplay(cm_norm, display_labels=['Normal','Fraud']).plot(
    ax=axes[1], colorbar=False, cmap='Purples')
axes[1].set_title('Confusion Matrix (normalised)', color='white')

plt.suptitle(f'Threshold = {api_threshold:.4f} (1st percentile of normal scores)', color=CYAN, fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'True  Negatives (correct normal): {tn:,}')
print(f'False Positives (false alarms)  : {fp:,}')
print(f'False Negatives (missed fraud)  : {fn:,}')
print(f'True  Positives (caught fraud)  : {tp:,}')
print()
print(classification_report(y_true, y_pred, target_names=['Normal', 'Fraud']))

## 11. Feature Importance Proxy

In [ ]:
importance = {}
for i, col in enumerate(FEATURE_COLS):
    normal_z = (normal[col].values - scaler.mean_[i]) / scaler.scale_[i]
    fraud_z  = (fraud[col].values  - scaler.mean_[i]) / scaler.scale_[i]
    importance[col] = abs(fraud_z.mean() - normal_z.mean())

imp_series = pd.Series(importance).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = [CYAN if v > imp_series.median() else PURPLE for v in imp_series.values]
ax.barh(imp_series.index, imp_series.values, color=colors, edgecolor='none', height=0.7)
ax.axvline(imp_series.median(), color=ORANGE, linestyle='--',
           label=f'Median: {imp_series.median():.2f}')
ax.set_xlabel('Mean Absolute Deviation (fraud vs normal, std units)')
ax.set_title('Feature Importance Proxy', color='white', pad=12)
ax.legend()

plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

print('Top 10 most discriminative features:')
print(imp_series.nlargest(10).to_string())

## 12. Summary

| Metric | Value |
|--------|-------|
| Dataset size | 284,807 transactions |
| Fraud rate | 0.172% (492 fraud) |
| Model | Isolation Forest (200 estimators) |
| Training data | Normal transactions only |
| Threshold | 1st percentile of normal training scores |

### Why Isolation Forest?
1. **No labelled fraud needed at training time** — we train on normal transactions only
2. **Handles high dimensionality** — 30 features, random subspace sampling keeps it efficient
3. **Fast inference** — suitable for real-time API use
4. **Explainable** — scores decompose per-feature (used in the API's `explanation` field)

### Threshold trade-offs
- **Lower threshold** → catch more fraud, more false alarms
- **Higher threshold** → fewer false alarms, more missed fraud
- Tune based on your business cost of false positives vs false negatives